# TurboQuant VRAM Test

Run on Colab with a GPU runtime. **Restart the runtime** after `pip install`
to ensure the new code is loaded.

Compares baseline vs TurboQuant KV cache memory. Each run measures its own
VRAM delta (peak − model loaded) to isolate KV cache + activation overhead
from model weights, avoiding cross-contamination between sequential runs.

In [ ]:
!pip install -q git+https://github.com/Echen1246/local-turboquant.git@triton-fused-attention

In [ ]:
import torch
import gc
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")
try:
    import triton
    print(f"Triton version: {triton.__version__}")
except ImportError:
    print("Triton not installed — will use PyTorch fallback attention")

In [ ]:
from turboquant import TurboQuantSession

MODEL = "meta-llama/Llama-3.1-8B-Instruct"
DTYPE = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"

PROMPT = "The quick brown fox " * 2000  # ~8K tokens
BITS = 4
MAX_NEW = 32

# Paper-accurate Q_prod: keys = (bits-1)-bit MSE + 1-bit QJL, values = bits-bit MSE
USE_QJL_KEYS = True
QUANTIZE_DECODE = True

print(f"Model: {MODEL}")
print(f"dtype: {DTYPE}")
print(f"Q_prod (QJL keys): {USE_QJL_KEYS}")
print(f"Quantize decode tokens: {QUANTIZE_DECODE}")
print(f"VRAM before loading: {torch.cuda.memory_allocated() / 1e9:.3f} GB")

In [ ]:
# --- Baseline ---
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

session_bl = TurboQuantSession.from_pretrained(
    MODEL, variant="baseline", bits=BITS,
    dtype=DTYPE, device_map="auto",
)

param_dtype_bl = next(session_bl.model.parameters()).dtype
param_gb_bl = sum(p.numel() * p.element_size() for p in session_bl.model.parameters()) / 1e9
print(f"Model params: {param_gb_bl:.2f} GB  (dtype={param_dtype_bl})")
if param_dtype_bl == torch.float32:
    print("⚠ WARNING: Model loaded in FP32! torch_dtype fix may not have taken effect.")
    print("  Try: Runtime → Restart runtime, then re-run all cells.")

model_loaded_bl = torch.cuda.memory_allocated()
torch.cuda.reset_peak_memory_stats()

text_bl = session_bl.generate(prompt=PROMPT, max_new_tokens=MAX_NEW)

peak_after_gen_bl = torch.cuda.max_memory_allocated()
gen_overhead_bl = peak_after_gen_bl - model_loaded_bl
telem_bl = session_bl.last_telemetry()

print(f"VRAM after model load: {model_loaded_bl / 1e9:.2f} GB")
print(f"Peak during generation: {peak_after_gen_bl / 1e9:.2f} GB")
print(f"Generation overhead (KV + activations): {gen_overhead_bl / 1e6:.0f} MB")
print(f"Gen time: {telem_bl.get('generation_seconds', 'N/A')}s")
print(f"Output: {text_bl[:200]}...")

del session_bl, text_bl
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# --- TurboQuant ---
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

session_tq = TurboQuantSession.from_pretrained(
    MODEL, variant="qmse_packed", bits=BITS,
    dtype=DTYPE, device_map="auto",
    use_qjl_keys=USE_QJL_KEYS,
    quantize_decode=QUANTIZE_DECODE,
)

param_dtype_tq = next(session_tq.model.parameters()).dtype
param_gb_tq = sum(p.numel() * p.element_size() for p in session_tq.model.parameters()) / 1e9
print(f"Model params: {param_gb_tq:.2f} GB  (dtype={param_dtype_tq})")

model_loaded_tq = torch.cuda.memory_allocated()
torch.cuda.reset_peak_memory_stats()

text_tq = session_tq.generate(prompt=PROMPT, max_new_tokens=MAX_NEW)

peak_after_gen_tq = torch.cuda.max_memory_allocated()
gen_overhead_tq = peak_after_gen_tq - model_loaded_tq
telem_tq = session_tq.last_telemetry()

print(f"VRAM after model load: {model_loaded_tq / 1e9:.2f} GB")
print(f"Peak during generation: {peak_after_gen_tq / 1e9:.2f} GB")
print(f"Generation overhead (KV + activations): {gen_overhead_tq / 1e6:.0f} MB")
print(f"Gen time: {telem_tq.get('generation_seconds', 'N/A')}s")
print(f"Packed KV: {telem_tq.get('packed_actual_bytes', 0) / 1e6:.0f} MB")
print(f"Dense KV would be: {telem_tq.get('dense_kv_bytes', 0) / 1e6:.0f} MB")
print(f"Payload savings: {telem_tq.get('payload_savings_percent', 0):.1f}%")

In [ ]:
# --- Comparison ---
dense_kv_mb = telem_tq.get('dense_kv_bytes', 0) / 1e6
packed_kv_mb = telem_tq.get('packed_actual_bytes', 0) / 1e6
payload_pct = telem_tq.get('payload_savings_percent', 0)
kv_saved_mb = dense_kv_mb - packed_kv_mb
overhead_saved_mb = (gen_overhead_bl - gen_overhead_tq) / 1e6

print("=" * 72)
print(f"{'Metric':<40} {'Baseline':>15} {'TurboQuant':>15}")
print("-" * 72)
print(f"{'Model dtype':<40} {str(param_dtype_bl):>15} {str(param_dtype_tq):>15}")
print(f"{'Model params (GB)':<40} {param_gb_bl:>15.2f} {param_gb_tq:>15.2f}")
print(f"{'Gen overhead: peak−model (MB)':<40} {gen_overhead_bl/1e6:>15.0f} {gen_overhead_tq/1e6:>15.0f}")
print(f"{'  → Overhead saved (MB)':<40} {'':>15} {overhead_saved_mb:>15.0f}")
print(f"{'Dense KV size (MB)':<40} {dense_kv_mb:>15.0f} {dense_kv_mb:>15.0f}")
print(f"{'Packed KV size (MB)':<40} {'—':>15} {packed_kv_mb:>15.0f}")
print(f"{'KV payload saved (MB)':<40} {'—':>15} {kv_saved_mb:>15.0f}")
print(f"{'KV payload savings %':<40} {'—':>15} {payload_pct:>14.1f}%")
print(f"{'Gen time (s)':<40} {telem_bl.get('generation_seconds','?'):>15} {telem_tq.get('generation_seconds','?'):>15}")
print("=" * 72)

if param_dtype_bl == torch.float32:
    print("\n⚠ Model is in FP32. Restart runtime and re-run to load in BF16.")
    print("  In BF16, model = ~15 GB and KV savings are a larger fraction of total VRAM.")
else:
    print(f"\n✓ Model correctly loaded in {param_dtype_bl}.")
    print(f"  KV cache is {dense_kv_mb:.0f} MB at this context length.")
    print(f"  TurboQuant saves {kv_saved_mb:.0f} MB ({payload_pct:.0f}%) of KV cache memory.")
    print(f"  For GB-scale savings, increase prompt to 32K+ tokens.")